In [8]:
pip install delta-spark==2.4.0

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, to_date, year
# CORREÇÃO: Adicionados IntegerType e DecimalType
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, DecimalType, IntegerType

# Caminho base do warehouse (DO SEU AMBIENTE)
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

# Criar SparkSession com Hive (COMO NO SEU SCRIPT)
spark = (
    SparkSession.builder
    .appName("Movies Silver ETL v2")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

# --- 1. EXTRAÇÃO (Extract) ---

# MUDANÇA 1: Ler budget e revenue como StringType para controlo total
schema = StructType([
    StructField("id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("release_date", StringType(), True), # Lemos como string primeiro
    StructField("genres", StringType(), True),
    StructField("belongs_to_collection", StringType(), True),
    StructField("original_language", StringType(), True),
    StructField("budget_musd", StringType(), True), # LIDO COMO STRING
    StructField("revenue_musd", StringType(), True), # LIDO COMO STRING
    StructField("production_companies", StringType(), True),
    StructField("production_countries", StringType(), True),
    StructField("vote_count", DoubleType(), True),
    StructField("vote_average", DoubleType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("runtime", DoubleType(), True),
    StructField("overview", StringType(), True),
    StructField("spoken_languages", StringType(), True),
    StructField("poster_path", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("cast_size", IntegerType(), True), 
    StructField("crew_size", IntegerType(), True), 
    StructField("director", StringType(), True)
])

# Caminho do ficheiro bronze (AJUSTE CONFORME NECESSÁRIO)
csv_path = "hdfs://hdfs-nn:9000/demo/bronze/movies_complete.csv"

# Ler o CSV de forma robusta
df = spark.read.csv(
    csv_path,
    header=True,
    schema=schema,
    multiLine=True,
    escape='"',
    quote='"',
    nullValue="",
    emptyValue=None
)

print(f"Total de registos lidos: {df.count()}")

# --- 2. TRANSFORMAÇÃO (Transform) ---

print("\n=== Início da Limpeza e Transformação ===")

# ==========================================================
# 2.1. TRATAMENTO (Filtro 'release_date')
# ==========================================================
df_clean = df.withColumn("release_date_str", trim(col("release_date")))
total_antes_filtro = df_clean.count()
df_clean = df_clean.filter(
    col("release_date_str").isNotNull() & (col("release_date_str") != "")
)
total_depois_filtro = df_clean.count()
print(f"TRATAMENTO (Filtro 'release_date'): Removidos {total_antes_filtro - total_depois_filtro} registos por data de estreia nula ou vazia.")

# ==========================================================
# 2.2. TRATAMENTO (Conversão de Data e Chave de Partição)
# ==========================================================
df_clean = df_clean.withColumn("release_date", to_date(col("release_date_str"), "yyyy-MM-dd"))
df_clean = df_clean.withColumn("release_year", year(col("release_date")))

total_antes_filtro_ano = df_clean.count()
df_clean = df_clean.filter(col("release_year").isNotNull())
total_depois_filtro_ano = df_clean.count()
if (total_antes_filtro_ano - total_depois_filtro_ano) > 0:
    print(f"TRATAMENTO (Filtro 'release_year'): Removidos {total_antes_filtro_ano - total_depois_filtro_ano} registos por formato de data inválido.")

# ==========================================================
# 2.3. TRATAMENTO (Transformação Financeira)
# MUDANÇA 2: Converter de String para Decimal, depois multiplicar, e formatar
# O cast para Decimal(38, 18) vai interpretar a notação "e" (ex: 2e-10) corretamente
# ==========================================================

# Definir um Decimal de alta precisão para a leitura
HighPrecDecimal = DecimalType(38, 18)
# Definir o Decimal final (formato dinheiro)
MoneyDecimal = DecimalType(38, 2)

df_clean = df_clean.withColumn(
    "budget_eur",
    (col("budget_musd").cast(HighPrecDecimal) * 1000000).cast(MoneyDecimal)
)
print("TRATAMENTO: Coluna 'budget_eur' criada (String -> Decimal -> * 1M -> 2 casas decimais).")

df_clean = df_clean.withColumn(
    "revenue_eur",
    (col("revenue_musd").cast(HighPrecDecimal) * 1000000).cast(MoneyDecimal)
)
print("TRATAMENTO: Coluna 'revenue_eur' criada (String -> Decimal -> * 1M -> 2 casas decimais).")


# ==========================================================
# 2.4. TRATAMENTO (NOVO FILTRO - Nulos Financeiros)
# ==========================================================
total_antes_filtro_fin = df_clean.count()
# Manter linhas onde PELO MENOS UM dos valores (budget ou revenue) NÃO é nulo
df_clean = df_clean.filter(
    col("budget_eur").isNotNull() | col("revenue_eur").isNotNull()
)
total_depois_filtro_fin = df_clean.count()
print(f"TRATAMENTO (Filtro Financeiro): Removidos {total_antes_filtro_fin - total_depois_filtro_fin} registos onde budget E revenue eram nulos.")


# ==========================================================
# 2.5. TRATAMENTO (Remoção e Seleção de Colunas)
# ==========================================================
colunas_removidas = [
    'id', 'tagline', 'genres', 'original_language', 'budget_musd', 'revenue_musd',
    'vote_count', 'vote_average', 'popularity', 'runtime', 'spoken_languages',
    'poster_path', 'cast', 'cast_size', 'crew_size', 'director',
    'release_date_str' # Coluna temporária
]

df_final = df_clean.drop(*colunas_removidas)

print(f"TRATAMENTO: Removidas {len(colunas_removidas)} colunas.")
print("\n=== Schema Final do DataFrame (o que será guardado) ===")
df_final.printSchema()


# --- 3. CARGA (Load) ---

silver_path = f"{warehouse_location}/silver.db/movies_silver"
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

print(f"\n=== A guardar dados limpos em {silver_path} (Formato Parquet) ===")

(df_final.write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("release_year")
    .save(silver_path)
)

print("Dados guardados em Parquet com sucesso.")

# Criar tabela Hive externa associada ao Parquet
spark.sql("DROP TABLE IF EXISTS silver.movies_silver")

print("A criar tabela Hive 'silver.movies_silver'...")
# MUDANÇA 3: Atualizar a tabela para usar DECIMAL e não DOUBLE
spark.sql(f"""
    CREATE EXTERNAL TABLE silver.movies_silver (
        title STRING,
        release_date DATE,
        belongs_to_collection STRING,
        production_companies STRING,
        production_countries STRING,
        overview STRING,
        budget_eur DECIMAL(38, 2),
        revenue_eur DECIMAL(38, 2)
    )
    PARTITIONED BY (release_year INT)
    STORED AS PARQUET
    LOCATION '{silver_path}'
""")

spark.catalog.recoverPartitions("silver.movies_silver")

print("\n✅ Tabela Parquet particionada criada com sucesso em silver.movies_silver")

# --- 4. VERIFICAÇÃO ---

print("\n=== Amostra de dados da tabela Hive (10 filmes) ===")
spark.sql("""
    SELECT release_year, title, budget_eur, revenue_eur
    FROM silver.movies_silver
    WHERE release_year > 2000
    LIMIT 10
""").show()

spark.stop()

Total de registos lidos: 44691

=== Início da Limpeza e Transformação ===
TRATAMENTO (Filtro 'release_date'): Removidos 34 registos por data de estreia nula ou vazia.
TRATAMENTO: Coluna 'budget_eur' criada (String -> Decimal -> * 1M -> 2 casas decimais).
TRATAMENTO: Coluna 'revenue_eur' criada (String -> Decimal -> * 1M -> 2 casas decimais).
TRATAMENTO (Filtro Financeiro): Removidos 33793 registos onde budget E revenue eram nulos.
TRATAMENTO: Removidas 17 colunas.

=== Schema Final do DataFrame (o que será guardado) ===
root
 |-- title: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- budget_eur: decimal(38,2) (nullable = true)
 |-- revenue_eur: decimal(38,2) (nullable = true)


=== A guardar dados limpos em hdfs://hdfs-nn:90

In [2]:
from pyspark.sql import SparkSession

# --- 1. CONFIGURAÇÃO DA SPARKSESSION (com Delta) ---
# Esta célula precisa de carregar as bibliotecas do Delta.
print("A iniciar a SparkSession com o pacote Delta Lake...")

# Define as variáveis do teu ambiente
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
hive_uris = "thrift://hive-metastore:9083"

spark = (
    SparkSession.builder
    .appName("Delta Converter (MoviesSilver)") # Um novo nome para a app
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", hive_uris)
    
    # --- Configuração essencial do Delta (COMPLETA) ---
    # 1. Diz ao Spark para fazer o download (ou encontrar) o pacote Delta
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") 
    # 2. Ativa as extensões SQL para comandos como 'USING DELTA' e 'CONVERT'
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    # 3. A LINHA CRÍTICA (que o erro anterior pediu):
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    .enableHiveSupport()
    .getOrCreate()
)

print("Sessão Spark com Delta pronta.")


# --- 2. O COMANDO DE CONVERSÃO ---

print("A converter a tabela Parquet 'silver.movies_silver' para Delta Lake...")

# Este comando "adota" os ficheiros Parquet existentes.
# É obrigatório incluir a definição da partição.
try:
    spark.sql("""
        CONVERT TO DELTA silver.movies_silver
        PARTITIONED BY (release_year INT)
    """)
    print("✅ Conversão da 'silver.movies_silver' concluída com sucesso!")

except Exception as e:
    print(f"❌ Ocorreu um erro durante a conversão.")
    print("Causas prováveis: 1) A tabela 'silver.movies_silver' não existe. 2) A tabela já é Delta.")
    print(e)


# (Opcional) Podes parar esta sessão
spark.stop()

A iniciar a SparkSession com o pacote Delta Lake...
Sessão Spark com Delta pronta.
A converter a tabela Parquet 'silver.movies_silver' para Delta Lake...
✅ Conversão da 'silver.movies_silver' concluída com sucesso!


In [4]:
from pyspark.sql import SparkSession

# Caminho base do warehouse (DO SEU AMBIENTE)
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

# --- 1. Conecta-te ao teu ambiente (COM CONFIGURAÇÃO DELTA) ---
print("A iniciar a SparkSession com o pacote Delta Lake...")
spark = (
    SparkSession.builder
    .appName("Analise de Filmes (Query Completa)")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    
    # --- MUDANÇA AQUI: Adicionada a configuração completa do Delta ---
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    .enableHiveSupport()
    .getOrCreate()
)
print("Sessão Spark com Delta iniciada.")

# --- 2. "Vê" todas as colunas que existem ---
print("\n=== A mostrar todas as colunas que existem na tabela 'silver.movies_silver' ===")
# O comando DESCRIBE mostra o nome e o tipo de dados de cada coluna
spark.sql("DESCRIBE silver.movies_silver").show()


# --- 3. "Vê" os dados de todas as colunas ---
print("\n=== A mostrar uma amostra de 10 filmes (todas as colunas) ===")
# O SELECT * vai buscar todas as colunas
spark.sql("""
    SELECT * FROM silver.movies_silver
    WHERE release_year = 2010
    LIMIT 10
""").show(truncate=False) # truncate=False evita que o texto (como 'overview') seja cortado

# Pára a sessão
spark.stop()
print("Sessão Spark parada.")

A iniciar a SparkSession com o pacote Delta Lake...
Sessão Spark com Delta iniciada.

=== A mostrar todas as colunas que existem na tabela 'silver.movies_silver' ===
+--------------------+-------------+-------+
|            col_name|    data_type|comment|
+--------------------+-------------+-------+
|               title|       string|   null|
|        release_date|         date|   null|
|belongs_to_collec...|       string|   null|
|production_companies|       string|   null|
|production_countries|       string|   null|
|            overview|       string|   null|
|          budget_eur|decimal(38,2)|   null|
|         revenue_eur|decimal(38,2)|   null|
|        release_year|          int|   null|
|# Partition Infor...|             |       |
|          # col_name|    data_type|comment|
|        release_year|          int|   null|
+--------------------+-------------+-------+


=== A mostrar uma amostra de 10 filmes (todas as colunas) ===
+-----------------------+------------+------------